# Day 3 - Conversational AI - aka Chatbot!

## The Use of Prompts

1. System Prompt - establish ground-rules, like "if you don't know the answer, just say so" provide critical background context

2. Context - during the conversation, insert context to give more relevant background information pertaining to the topic

3. Multi-Shot Prompting - provide example conversations to prime for specific scenarios, train on conversational style amnd demonstrate complex interactions

In [23]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [24]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"OpenAI API Key exists and begins {groq_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins gsk_OyAI


In [25]:
# Initialize
groq_url = "https://api.groq.com/openai/v1"
MODEL="llama-3.3-70b-versatile"

groq = OpenAI(api_key = groq_api_key, base_url = groq_url)

In [26]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant"

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [27]:
def chat(message, history):
    return "bananas"

In [28]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [29]:
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [30]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


## OK! Let's write a slightly better chat callback!

In [31]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [32]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [33]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = groq.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [34]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


## OK let's keep going!

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [35]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [36]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [37]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [38]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


In [39]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = groq.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [40]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


In [41]:
system_message += "\nIf the customer asks for kids section, you should respond that kids section is not available till now but we will add very soon, \
but remind the customer to look at hats!"

In [42]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Conversational Assistants are of course a hugely common use case for Gen AI, and the latest frontier models are remarkably good at nuanced conversation. And Gradio makes it easy to have a user interface. Another crucial skill we covered is how to use prompting to provide context, information and examples.
<br/><br/>
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.</span>
        </td>
    </tr>
</table>

## AI Shopping Assistant

In [43]:
products = [
    {"id": 1, "name": "Nike Air Zoom Pegasus", "price": 6500, "category": "shoes", "brand": "Nike", "rating": 4.5, "discount": 20, "stock": 12},
    {"id": 2, "name": "Adidas Ultraboost 22", "price": 9000, "category": "shoes", "brand": "Adidas", "rating": 4.7, "discount": 15, "stock": 8},
    {"id": 3, "name": "Puma Running Pro", "price": 4000, "category": "shoes", "brand": "Puma", "rating": 4.2, "discount": 25, "stock": 20},
    {"id": 4, "name": "Bata Formal Leather Shoes", "price": 2500, "category": "shoes", "brand": "Bata", "rating": 4.0, "discount": 10, "stock": 15},

    {"id": 5, "name": "Nike Sports Cap", "price": 1200, "category": "hats", "brand": "Nike", "rating": 4.3, "discount": 40, "stock": 25},
    {"id": 6, "name": "Adidas Baseball Cap", "price": 1000, "category": "hats", "brand": "Adidas", "rating": 4.4, "discount": 35, "stock": 18},
    {"id": 7, "name": "Puma Snapback Hat", "price": 900, "category": "hats", "brand": "Puma", "rating": 4.1, "discount": 50, "stock": 10},

    {"id": 8, "name": "Levi's Slim Fit Jeans", "price": 3000, "category": "clothing", "brand": "Levi's", "rating": 4.6, "discount": 20, "stock": 14},
    {"id": 9, "name": "H&M Casual T-Shirt", "price": 800, "category": "clothing", "brand": "H&M", "rating": 4.2, "discount": 30, "stock": 30},
    {"id": 10, "name": "Zara Formal Shirt", "price": 2200, "category": "clothing", "brand": "Zara", "rating": 4.5, "discount": 15, "stock": 9},

    {"id": 11, "name": "Fossil Analog Watch", "price": 7000, "category": "accessories", "brand": "Fossil", "rating": 4.7, "discount": 25, "stock": 6},
    {"id": 12, "name": "Casio G-Shock", "price": 5000, "category": "accessories", "brand": "Casio", "rating": 4.8, "discount": 10, "stock": 11},
    {"id": 13, "name": "Ray-Ban Sunglasses", "price": 6000, "category": "accessories", "brand": "Ray-Ban", "rating": 4.6, "discount": 20, "stock": 7},

    {"id": 14, "name": "Wildcraft Backpack", "price": 1800, "category": "bags", "brand": "Wildcraft", "rating": 4.3, "discount": 35, "stock": 22},
    {"id": 15, "name": "Skybags Travel Bag", "price": 2500, "category": "bags", "brand": "Skybags", "rating": 4.4, "discount": 30, "stock": 16},

    {"id": 16, "name": "Woodland Leather Belt", "price": 1500, "category": "belt", "brand": "Woodland", "rating": 4.2, "discount": 5, "stock": 13},
    {"id": 17, "name": "Titan Formal Belt", "price": 1200, "category": "belt", "brand": "Titan", "rating": 4.1, "discount": 10, "stock": 19},
]

In [44]:
def find_products(user_message):
    msg = user_message.lower()
    results = []

    for p in products:
        if p["category"] in msg or p["brand"].lower() in msg:
            results.append(p)

    results = sorted(results, key=lambda x: (-x["rating"], -x["discount"]))

    return results[:5]

In [45]:
user_memory = {}

def update_memory(message):
    msg = message.lower()

    if "cheap" in msg or "budget" in msg:
        user_memory["price"] = "low"
    elif "premium" in msg or "expensive" in msg:
        user_memory["price"] = "high"

In [46]:
def detect_intent(message):
    msg = message.lower()

    if "buy" in msg or "recommend" in msg:
        return "shopping"
    elif "price" in msg:
        return "price_query"
    else:
        return "general"

In [50]:
def chat(message, history):
    update_memory(message)

    relevant_products = find_products(message)

    # Apply memory filter
    if user_memory.get("price") == "low":
        relevant_products = sorted(relevant_products, key=lambda x: x["price"])
    elif user_memory.get("price") == "high":
        relevant_products = sorted(relevant_products, key=lambda x: -x["price"])

    # Format product info
    product_info = ""
    for p in relevant_products:
        product_info += (
            f"{p['name']} ({p['brand']})\n"
            f"Price: ₹{p['price']} | Discount: {p['discount']}% | Rating: ⭐{p['rating']}\n"
            f"Stock left: {p['stock']}\n\n"
        )

    system_message = f"""
    You are a smart AI shopping assistant.

    Available products:
    {product_info}

    Rules:
    - Recommend products based on user needs
    - Highlight discounts and ratings
    - Create urgency using stock info
    - Be friendly and persuasive
    """

    messages = [{"role": "system", "content": system_message}]

    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    messages.append({"role": "user", "content": message})

    stream = groq.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""
    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        response += content
        yield response

In [51]:
gr.ChatInterface(
    fn=chat,
    title="🛍️ AI Shopping Assistant",
    description="Ask me to recommend products, compare items, or find deals!",
    theme="soft"
).launch()

c:\Users\KIIT\projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.
